# GPU: RNN và LSTM

Simple RNN và LSTM nhỏ, đầu vào chỉ là chuỗi target (F0), dự báo một bước rồi recursive. Hai baseline mùa vụ là đối chứng. Dùng GPU Colab; không huấn luyện local.

Mặc định chỉ chạy F5, H=30/548, G/COGS. Chuyển MODE sang cv để chạy F1-F5. Không gọi kết quả này là benchmark đủ 40 model.

## 1. Kiểm runtime và cài thư viện

Chạy theo thứ tự. Nếu Colab yêu cầu restart sau pip, restart rồi chạy lại từ đầu.

In [ ]:
import os, sys
from pathlib import Path
if not os.getenv("COLAB_RELEASE_TAG") or not Path("/content").is_dir():
    raise RuntimeError("Chỉ chạy trên Google Colab hosted runtime, không chạy local.")
if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f"Notebook dùng Colab Python 3.13; hiện là {sys.version}.")
print("Runtime:", sys.version)


In [ ]:
import subprocess, sys
packages = ["numpy==2.1.3", "pandas==2.2.3", "scikit-learn==1.6.1",
            "statsmodels==0.14.4", "matplotlib==3.9.4", "joblib==1.4.2"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--only-binary=:all:", *packages])
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chọn Runtime > Change runtime type > GPU rồi chạy lại notebook.")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))


## 2. Tạo runner độc lập trên VM Colab

Code và cấu hình được nhúng trong notebook; không cần clone repo.

In [ ]:
import json, os, sys
from pathlib import Path
project = Path("/content/retail-sales-forecasting-research")
project.mkdir(exist_ok=True)
embedded_files = json.loads('{"cloud_pilot.py": "\\"\\"\\"Pilot CPU chỉ chạy trên Colab/Databricks; không dùng code hệ thống chính.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport time\\nimport warnings\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nDEVELOPMENT_END = pd.Timestamp(\\"2021-07-01\\")\\n\\n\\ndef cloud_runtime():\\n    # Guard chống chạy nhầm; đây không phải cơ chế chứng thực hạ tầng.\\n    if os.getenv(\\"COLAB_RELEASE_TAG\\") and Path(\\"/content\\").is_dir():\\n        return \\"colab\\"\\n    if os.getenv(\\"DATABRICKS_RUNTIME_VERSION\\"):\\n        return \\"databricks\\"\\n    raise RuntimeError(\\"Không train local. Chạy trên runtime Colab hoặc Databricks.\\")\\n\\n\\ndef sha256(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_suffix(path.suffix + \\".tmp\\")\\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,\\n                                    allow_nan=False), encoding=\\"utf-8\\")\\n    temporary.replace(path)\\n\\n\\ndef validate_data(frame):\\n    required = {\\"sale_date\\", \\"g\\", \\"cogs\\"}\\n    if not required.issubset(frame.columns):\\n        raise ValueError(\\"CSV phải có sale_date,g,cogs từ rpt_forecast_actual.\\")\\n    data = frame[list(sorted(required))].copy()\\n    data[\\"sale_date\\"] = pd.to_datetime(data[\\"sale_date\\"], errors=\\"raise\\")\\n    dates = data[\\"sale_date\\"]\\n    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():\\n        raise ValueError(\\"sale_date phải là ngày không có timezone/giờ.\\")\\n    data = data.sort_values(\\"sale_date\\").reset_index(drop=True)\\n    expected = pd.date_range(\\"2012-07-04\\", \\"2022-12-31\\", freq=\\"D\\")\\n    if not pd.DatetimeIndex(data[\\"sale_date\\"]).equals(expected):\\n        raise ValueError(\\"Lịch actual không khớp 3833 ngày đã đăng ký; kiểm lại snapshot.\\")\\n    for name in (\\"g\\", \\"cogs\\"):\\n        data[name] = pd.to_numeric(data[name], errors=\\"raise\\")\\n        if not np.isfinite(data[name]).all() or (data[name] < 0).any():\\n            raise ValueError(\\"Target thiếu, âm hoặc không hữu hạn; cần điều tra dữ liệu.\\")\\n    return data\\n\\n\\ndef load_snapshot(csv_path, manifest_path):\\n    manifest = json.loads(Path(manifest_path).read_text(encoding=\\"utf-8\\"))\\n    for key in (\\"snapshot_id\\", \\"release_id\\", \\"source_relation\\", \\"exported_at\\",\\n                \\"sha256\\", \\"actual_only\\"):\\n        if key not in manifest:\\n            raise ValueError(\\"Manifest thiếu \\" + key)\\n    if manifest[\\"actual_only\\"] is not True:\\n        raise ValueError(\\"Manifest chưa xác nhận actual_only.\\")\\n    relation_parts = manifest[\\"source_relation\\"].split(\\".\\")\\n    if (len(relation_parts) != 3 or relation_parts[-1] != \\"rpt_forecast_actual\\"\\n            or relation_parts[-2] != \\"r_\\" + manifest[\\"release_id\\"] + \\"_reporting\\"):\\n        raise ValueError(\\"Manifest phải chỉ đúng rpt_forecast_actual của release.\\")\\n    if sha256(csv_path) != manifest[\\"sha256\\"]:\\n        raise ValueError(\\"Checksum CSV khác manifest.\\")\\n    return validate_data(pd.read_csv(csv_path)), manifest\\n\\n\\ndef score(y, prediction, train):\\n    y, prediction, train = [np.asarray(v, dtype=float) for v in (y, prediction, train)]\\n    if y.shape != prediction.shape or y.size == 0:\\n        raise ValueError(\\"Prediction và actual phải cùng shape, không rỗng.\\")\\n    if not all(np.isfinite(v).all() for v in (y, prediction, train)):\\n        raise ValueError(\\"Metric không nhận NaN/Inf.\\")\\n    error = prediction - y\\n    mae = float(np.abs(error).mean())\\n    rse_den = float(np.square(y - y.mean()).sum())\\n    mase_den = float(np.abs(train[365:] - train[:-365]).mean()) if len(train) > 365 else 0\\n    wape_den = float(np.abs(y).sum())\\n    nonzero = y != 0\\n    return {\\n        \\"n\\": len(y), \\"mae\\": mae, \\"rmse\\": float(np.sqrt(np.square(error).mean())),\\n        \\"rse\\": float(np.square(error).sum() / rse_den) if rse_den else None,\\n        \\"mape\\": float(100 * np.mean(np.abs(error / y))) if nonzero.all() else None,\\n        \\"mape_nonzero\\": float(100 * np.mean(np.abs(error[nonzero] / y[nonzero]))) if nonzero.any() else None,\\n        \\"mape_coverage\\": float(nonzero.mean()),\\n        \\"mase\\": mae / mase_den if mase_den else None,\\n        \\"wape\\": float(100 * np.abs(error).sum() / wape_den) if wape_den else None,\\n        \\"bias\\": float(error.mean()), \\"total_error\\": float(error.sum()),\\n    }\\n\\n\\ndef calendar(dates):\\n    dates = pd.DatetimeIndex(dates)\\n    result = pd.DataFrame(index=dates)\\n    # Mã hóa cố định, không học từ validation.\\n    for month in range(2, 13):\\n        result[\\"month_\\" + str(month)] = (dates.month == month).astype(float)\\n    result[\\"day\\"] = dates.day / 31.0\\n    result[\\"days_in_month\\"] = dates.days_in_month / 31.0\\n    result[\\"days_to_end\\"] = (dates.days_in_month - dates.day) / 31.0\\n    phase = 2 * np.pi * (dates.dayofyear - 1) / np.where(dates.is_leap_year, 366, 365)\\n    for harmonic in (1, 2, 3):\\n        result[\\"sin_\\" + str(harmonic)] = np.sin(harmonic * phase)\\n        result[\\"cos_\\" + str(harmonic)] = np.cos(harmonic * phase)\\n    return result\\n\\n\\ndef history_row(values, config):\\n    row = [values[-lag] for lag in config[\\"lag_days\\"]]\\n    for window in config[\\"rolling_days\\"]:\\n        row.extend([np.mean(values[-window:]), np.std(values[-window:], ddof=0)])\\n    return row\\n\\n\\ndef supervised(train, config):\\n    first = max(config[\\"lag_days\\"] + config[\\"rolling_days\\"])\\n    values = train.to_numpy(dtype=float)\\n    if len(values) <= first:\\n        raise ValueError(\\"Không đủ train cho lag đã đăng ký.\\")\\n    cal = calendar(train.index).to_numpy()\\n    rows = [history_row(values[:i], config) + list(cal[i]) for i in range(first, len(values))]\\n    return np.asarray(rows), values[first:], first\\n\\n\\ndef baseline(train, future, model):\\n    values = train.to_numpy(dtype=float)\\n    if model == \\"mean\\":\\n        return np.repeat(values.mean(), len(future))\\n    if model == \\"naive\\":\\n        return np.repeat(values[-1], len(future))\\n    if model == \\"drift\\":\\n        return values[-1] + np.arange(1, len(future) + 1) * (values[-1] - values[0]) / (len(values) - 1)\\n    known = train.to_dict()\\n    predictions = []\\n    for date in future:\\n        if model == \\"snaive7\\":\\n            reference = date - pd.Timedelta(days=7)\\n        else:\\n            years = 1 if model == \\"snaive_year\\" else 2\\n            reference = date - pd.DateOffset(years=years)\\n        value = float(known[reference])\\n        predictions.append(value)\\n        known[date] = max(value, 0.0)\\n    return np.asarray(predictions)\\n\\n\\ndef fit_predict(train, future, model_name, config):\\n    cloud_runtime()  # Kiểm trước mọi đường fit.\\n    start = time.perf_counter()\\n    fitted = None\\n    fitted_dates = None\\n    metadata = {\\"input_set\\": \\"F0\\", \\"strategy\\": \\"native\\", \\"point\\": \\"point\\"}\\n    estimator = None\\n    baseline_names = {\\"mean\\", \\"naive\\", \\"drift\\", \\"snaive7\\", \\"snaive_year\\", \\"snaive_two_year\\"}\\n    if model_name in baseline_names:\\n        fit_seconds = time.perf_counter() - start\\n        pred_start = time.perf_counter()\\n        prediction = baseline(train, future, model_name)\\n        estimator = {\\"method\\": model_name, \\"history\\": train.to_dict()}\\n    elif model_name in {\\"ridge\\", \\"lightgbm\\"}:\\n        from sklearn.linear_model import Ridge\\n        from sklearn.pipeline import make_pipeline\\n        from sklearn.preprocessing import StandardScaler\\n        features, labels, first = supervised(train, config)\\n        feature_seconds = time.perf_counter() - start\\n        start = time.perf_counter()\\n        if model_name == \\"ridge\\":\\n            estimator = make_pipeline(StandardScaler(), Ridge(**config[\\"ridge\\"]))\\n        else:\\n            from lightgbm import LGBMRegressor\\n            estimator = LGBMRegressor(random_state=config[\\"seed\\"], **config[\\"lightgbm\\"])\\n        estimator.fit(features, labels)\\n        fit_seconds = time.perf_counter() - start\\n        fitted = estimator.predict(features)\\n        fitted_dates = train.index[first:]\\n        history = list(train.to_numpy(dtype=float))\\n        cal = calendar(future).to_numpy()\\n        pred_start = time.perf_counter()\\n        prediction = []\\n        for i in range(len(future)):\\n            row = np.asarray([history_row(history, config) + list(cal[i])])\\n            value = float(estimator.predict(row)[0])\\n            prediction.append(value)\\n            history.append(max(value, 0.0))\\n        prediction = np.asarray(prediction)\\n        metadata.update(input_set=\\"F1\\", strategy=\\"recursive\\", feature_seconds=feature_seconds)\\n    elif model_name == \\"log_sarimax\\":\\n        from statsmodels.tsa.statespace.sarimax import SARIMAX\\n        options = config[\\"log_sarimax\\"]\\n        with warnings.catch_warnings(record=True) as observed:\\n            warnings.simplefilter(\\"always\\")\\n            estimator = SARIMAX(np.log1p(train.to_numpy()), exog=calendar(train.index).to_numpy(),\\n                                order=tuple(options[\\"order\\"]), trend=options[\\"trend\\"],\\n                                enforce_stationarity=True, enforce_invertibility=True).fit(\\n                                    disp=False, maxiter=options[\\"maxiter\\"])\\n        metadata[\\"warnings\\"] = [str(w.message) for w in observed]\\n        if not estimator.mle_retvals.get(\\"converged\\", False):\\n            raise RuntimeError(\\"SARIMAX không hội tụ; ghi failed, không nhận score.\\")\\n        fit_seconds = time.perf_counter() - start\\n        fitted = np.expm1(np.asarray(estimator.fittedvalues))\\n        fitted_dates = train.index\\n        pred_start = time.perf_counter()\\n        prediction = np.expm1(np.asarray(estimator.forecast(len(future), exog=calendar(future).to_numpy())))\\n        metadata.update(input_set=\\"F1\\", point=\\"inverse_log_location\\", strategy=\\"native\\")\\n    else:\\n        raise ValueError(\\"Model chưa triển khai: \\" + model_name)\\n    pred_seconds = time.perf_counter() - pred_start\\n    if not np.isfinite(prediction).all():\\n        raise ValueError(\\"Prediction không hữu hạn.\\")\\n    return prediction, fitted, fitted_dates, estimator, {\\n        **metadata, \\"fit_seconds\\": fit_seconds, \\"predict_seconds\\": pred_seconds,\\n    }\\n\\n\\ndef make_figure(train, actual, prediction, fitted, fitted_dates, path, title):\\n    import matplotlib\\n    matplotlib.use(\\"Agg\\")\\n    import matplotlib.pyplot as plt\\n    fig, axes = plt.subplots(2, 1, figsize=(12, 7))\\n    axes[0].plot(train.index, train.values, label=\\"Actual train\\", linewidth=0.8)\\n    if fitted is not None:\\n        axes[0].plot(fitted_dates, np.maximum(fitted, 0), label=\\"Fitted (in-sample)\\", alpha=0.7, linewidth=0.7)\\n    axes[0].set_title(\\"Train — chỉ dùng chẩn đoán\\")\\n    axes[1].plot(actual.index, actual.values, label=\\"Actual validation\\")\\n    axes[1].plot(actual.index, prediction, label=\\"Forecast từ origin cố định\\")\\n    axes[1].set_title(title + \\" — development F5, không phải final test\\")\\n    for axis in axes:\\n        axis.set_ylabel(\\"Giá trị tiền theo nguồn\\")\\n        axis.legend()\\n    fig.tight_layout()\\n    fig.savefig(path, dpi=180)\\n    plt.close(fig)\\n\\n\\ndef run(csv_path, manifest_path, config_path, output):\\n    runtime = cloud_runtime()\\n    data, source = load_snapshot(csv_path, manifest_path)\\n    config = json.loads(Path(config_path).read_text(encoding=\\"utf-8\\"))\\n    origin = pd.Timestamp(config[\\"origin\\"])\\n    if origin != pd.Timestamp(\\"2019-12-31\\") or config[\\"horizons\\"] != [30, 548]:\\n        raise ValueError(\\"Pilot chỉ chạy F5, H=30/548; không mở final test.\\")\\n    if config[\\"targets\\"] != [\\"G\\", \\"COGS\\"] or len(set(config[\\"models\\"])) != len(config[\\"models\\"]):\\n        raise ValueError(\\"Config target/model không hợp lệ.\\")\\n    if origin + pd.Timedelta(days=max(config[\\"horizons\\"])) > DEVELOPMENT_END:\\n        raise ValueError(\\"Validation vượt development.\\")\\n    packages = {}\\n    for name in (\\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"lightgbm\\", \\"statsmodels\\", \\"matplotlib\\", \\"joblib\\"):\\n        packages[name] = importlib.metadata.version(name)\\n    identity = {\\n        \\"snapshot_sha256\\": source[\\"sha256\\"], \\"config_sha256\\": sha256(config_path),\\n        \\"code_sha256\\": sha256(__file__), \\"runtime\\": runtime,\\n        \\"python\\": platform.python_version(), \\"packages\\": packages,\\n        \\"machine\\": platform.machine(), \\"cpu_count\\": os.cpu_count(),\\n    }\\n    run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]\\n    root = Path(output) / run_id\\n    root.mkdir(parents=True, exist_ok=True)\\n    run_manifest = {\\"run_id\\": run_id, \\"source\\": source, \\"config\\": config, **identity}\\n    write_json(root / \\"manifest.json\\", run_manifest)\\n    actual_data = data.set_index(\\"sale_date\\")\\n    import joblib\\n    results = []\\n    for target in config[\\"targets\\"]:\\n        series = actual_data[\\"g\\" if target == \\"G\\" else \\"cogs\\"]\\n        train = series.loc[:origin].copy()\\n        for horizon in config[\\"horizons\\"]:\\n            dates = pd.date_range(origin + pd.Timedelta(days=1), periods=horizon)\\n            actual = series.reindex(dates)\\n            for model in config[\\"models\\"]:\\n                task = root / (target + \\"_\\" + str(horizon) + \\"_\\" + model)\\n                task.mkdir(exist_ok=True)\\n                status_path = task / \\"status.json\\"\\n                if status_path.exists():\\n                    old = json.loads(status_path.read_text(encoding=\\"utf-8\\"))\\n                    hashes = old.get(\\"artifact_sha256\\", {})\\n                    if old.get(\\"status\\") == \\"success\\" and hashes and all(\\n                            (task / name).exists() and sha256(task / name) == digest\\n                            for name, digest in hashes.items()):\\n                        results.append(old)\\n                        continue\\n                record = {\\n                    \\"run_id\\": run_id, \\"snapshot_id\\": source[\\"snapshot_id\\"], \\"model_id\\": model,\\n                    \\"target\\": target, \\"origin\\": str(origin.date()), \\"horizon\\": horizon,\\n                    \\"fold_id\\": \\"F5\\", \\"split\\": \\"development\\", \\"seed\\": config[\\"seed\\"],\\n                }\\n                write_json(status_path, {**record, \\"status\\": \\"running\\"})\\n                try:\\n                    raw, fitted, fitted_dates, estimator, timing = fit_predict(train, dates, model, config)\\n                    prediction = np.maximum(raw, 0)\\n                    predictions = pd.DataFrame({\\n                        **{key: [value] * horizon for key, value in record.items()},\\n                        \\"ds\\": dates, \\"lead_time\\": np.arange(1, horizon + 1),\\n                        \\"y_true\\": actual.to_numpy(), \\"y_pred_raw\\": raw, \\"y_pred\\": prediction,\\n                    })\\n                    predictions.to_csv(task / \\"predictions.csv\\", index=False)\\n                    joblib.dump(estimator, task / \\"model.joblib\\")\\n                    if fitted is not None:\\n                        pd.DataFrame({\\"ds\\": fitted_dates, \\"fitted_raw\\": fitted}).to_csv(task / \\"fitted.csv\\", index=False)\\n                    make_figure(train, actual, prediction, fitted, fitted_dates, task / \\"plot.png\\",\\n                                target + \\" / \\" + model + \\" / H=\\" + str(horizon))\\n                    artifact_names = [\\"predictions.csv\\", \\"model.joblib\\", \\"plot.png\\"]\\n                    if fitted is not None:\\n                        artifact_names.append(\\"fitted.csv\\")\\n                    record.update(status=\\"success\\", **score(actual.to_numpy(), prediction, train.to_numpy()),\\n                                  **timing, clipped_negative=int((raw < 0).sum()),\\n                                  artifact_sha256={name: sha256(task / name) for name in artifact_names})\\n                except Exception as exc:\\n                    record.update(status=\\"failed\\", error_type=type(exc).__name__, error=str(exc))\\n                write_json(status_path, record)\\n                results.append(record)\\n                print(target, horizon, model, record[\\"status\\"], flush=True)\\n                pd.DataFrame(results).drop(columns=[\\"artifact_sha256\\"], errors=\\"ignore\\").to_csv(\\n                    root / \\"metrics.csv\\", index=False)\\n    pd.DataFrame(results).drop(columns=[\\"artifact_sha256\\"], errors=\\"ignore\\").to_csv(root / \\"metrics.csv\\", index=False)\\n    write_json(root / \\"summary.json\\", {\\"total\\": len(results),\\n                                     \\"success\\": sum(r[\\"status\\"] == \\"success\\" for r in results),\\n                                     \\"failed\\": sum(r[\\"status\\"] != \\"success\\" for r in results)})\\n    return str(root)\\n\\n\\nif __name__ == \\"__main__\\":\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--csv\\", required=True)\\n    parser.add_argument(\\"--manifest\\", required=True)\\n    parser.add_argument(\\"--config\\", default=\\"configs/pilot_cpu.json\\")\\n    parser.add_argument(\\"--output\\", required=True)\\n    args = parser.parse_args()\\n    print(run(args.csv, args.manifest, args.config, args.output))\\n", "screening.py": "\\"\\"\\"Screening riêng cho các họ model người dùng yêu cầu; chỉ fit trên cloud.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport time\\nimport warnings\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom cloud_pilot import (\\n    DEVELOPMENT_END, calendar, cloud_runtime, history_row, load_snapshot,\\n    score, sha256, supervised, write_json,\\n)\\n\\nFOLDS = {\\n    \\"F1\\": \\"2017-12-31\\",\\n    \\"F2\\": \\"2018-06-30\\",\\n    \\"F3\\": \\"2018-12-31\\",\\n    \\"F4\\": \\"2019-06-30\\",\\n    \\"F5\\": \\"2019-12-31\\",\\n}\\nTABULAR = {\\"random_forest\\", \\"extra_trees\\", \\"xgboost\\", \\"lightgbm\\", \\"catboost\\"}\\nSTATISTICAL = {\\"arima\\", \\"arimax\\", \\"sarima\\", \\"sarimax\\"}\\nNEURAL = {\\"rnn\\", \\"lstm\\"}\\nSUPPORTED = TABULAR | STATISTICAL | NEURAL | {\\"snaive_year\\", \\"snaive_two_year\\"}\\n\\n\\ndef validate_config(config):\\n    models = config[\\"models\\"]\\n    folds = config[\\"folds\\"]\\n    horizons = config[\\"horizons\\"]\\n    if not models or len(models) != len(set(models)) or set(models) - SUPPORTED:\\n        raise ValueError(\\"Danh sách model trống, trùng hoặc chưa triển khai.\\")\\n    if not folds or len(folds) != len(set(folds)) or set(folds) - FOLDS.keys():\\n        raise ValueError(\\"Fold không hợp lệ.\\")\\n    if config[\\"targets\\"] != [\\"G\\", \\"COGS\\"]:\\n        raise ValueError(\\"Target phải là G và COGS.\\")\\n    if not horizons or len(horizons) != len(set(horizons)) or any(h not in (7, 30, 90, 365, 548) for h in horizons):\\n        raise ValueError(\\"Horizon không hợp lệ.\\")\\n    if set(models) & NEURAL and config.get(\\"runtime\\") != \\"gpu\\":\\n        raise ValueError(\\"RNN/LSTM chỉ chạy trong cấu hình GPU.\\")\\n    if set(models) & (TABULAR | STATISTICAL) and config.get(\\"runtime\\") != \\"cpu\\":\\n        raise ValueError(\\"CPU model phải dùng cấu hình CPU.\\")\\n    for fold in folds:\\n        if pd.Timestamp(FOLDS[fold]) + pd.Timedelta(days=max(horizons)) > DEVELOPMENT_END:\\n            raise ValueError(\\"Validation vượt development.\\")\\n    if config.get(\\"seed\\") is None:\\n        raise ValueError(\\"Thiếu seed.\\")\\n\\n\\ndef fit_tabular(train, future, model_name, config):\\n    from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor\\n    features, labels, first = supervised(train, config)\\n    start = time.perf_counter()\\n    options = config[model_name].copy()\\n    seed = int(config[\\"seed\\"])\\n    if model_name == \\"random_forest\\":\\n        estimator = RandomForestRegressor(random_state=seed, **options)\\n    elif model_name == \\"extra_trees\\":\\n        estimator = ExtraTreesRegressor(random_state=seed, **options)\\n    elif model_name == \\"xgboost\\":\\n        from xgboost import XGBRegressor\\n        estimator = XGBRegressor(random_state=seed, **options)\\n    elif model_name == \\"lightgbm\\":\\n        from lightgbm import LGBMRegressor\\n        estimator = LGBMRegressor(random_state=seed, **options)\\n    else:\\n        from catboost import CatBoostRegressor\\n        estimator = CatBoostRegressor(random_seed=seed, **options)\\n    estimator.fit(features, labels)\\n    fit_seconds = time.perf_counter() - start\\n    fitted = estimator.predict(features)\\n    history = list(train.to_numpy(dtype=float))\\n    cal = calendar(future).to_numpy()\\n    start = time.perf_counter()\\n    prediction = []\\n    for row in cal:\\n        x = np.asarray([history_row(history, config) + list(row)])\\n        value = float(estimator.predict(x)[0])\\n        prediction.append(value)\\n        history.append(max(0.0, value))\\n    return np.asarray(prediction), fitted, train.index[first:], estimator, {\\n        \\"input_set\\": \\"F1\\", \\"strategy\\": \\"recursive\\",\\n        \\"fit_seconds\\": fit_seconds, \\"predict_seconds\\": time.perf_counter() - start,\\n    }\\n\\n\\n\\ndef statistical_exog(dates):\\n    columns = [f\\"{kind}_{harmonic}\\" for harmonic in (1, 2, 3)\\n               for kind in (\\"sin\\", \\"cos\\")]\\n    return calendar(dates)[columns].to_numpy()\\n\\n\\ndef fit_statistical(train, future, model_name, config):\\n    from statsmodels.tsa.statespace.sarimax import SARIMAX\\n    options = config[\\"statistical\\"][model_name]\\n    use_exog = model_name in {\\"arimax\\", \\"sarimax\\"}\\n    seasonal = model_name in {\\"sarima\\", \\"sarimax\\"}\\n    exog_train = statistical_exog(train.index) if use_exog else None\\n    exog_future = statistical_exog(future) if use_exog else None\\n    seasonal_order = tuple(options[\\"seasonal_order\\"]) if seasonal else (0, 0, 0, 0)\\n    start = time.perf_counter()\\n    with warnings.catch_warnings(record=True) as observed:\\n        warnings.simplefilter(\\"always\\")\\n        estimator = SARIMAX(\\n            np.log1p(train.to_numpy(dtype=float)),\\n            exog=exog_train,\\n            order=tuple(options[\\"order\\"]),\\n            seasonal_order=seasonal_order,\\n            trend=options.get(\\"trend\\", \\"n\\"),\\n            enforce_stationarity=False,\\n            enforce_invertibility=False,\\n        ).fit(disp=False, maxiter=options[\\"maxiter\\"])\\n    if not estimator.mle_retvals.get(\\"converged\\", False):\\n        raise RuntimeError(model_name + \\" không hội tụ sau \\" + str(options[\\"maxiter\\"]) + \\" vòng.\\")\\n    fit_seconds = time.perf_counter() - start\\n    fitted = np.expm1(np.asarray(estimator.fittedvalues))\\n    start = time.perf_counter()\\n    prediction = np.expm1(np.asarray(estimator.forecast(len(future), exog=exog_future)))\\n    return prediction, fitted, train.index, estimator, {\\n        \\"input_set\\": \\"F1\\" if use_exog else \\"F0\\",\\n        \\"strategy\\": \\"native\\", \\"transform\\": \\"log1p_inverse_location\\",\\n        \\"seasonal_order\\": list(seasonal_order),\\n        \\"exog\\": \\"annual_fourier_3\\" if use_exog else None,\\n        \\"warning_count\\": len(observed),\\n        \\"fit_seconds\\": fit_seconds, \\"predict_seconds\\": time.perf_counter() - start,\\n    }\\n\\n\\ndef fit_neural(train, future, model_name, config):\\n    import torch\\n    from torch import nn\\n    from torch.utils.data import DataLoader, TensorDataset\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"Cấu hình GPU yêu cầu Colab GPU runtime có CUDA.\\")\\n    options = config[\\"neural\\"]\\n    seed = int(config[\\"seed\\"])\\n    torch.manual_seed(seed)\\n    torch.cuda.manual_seed_all(seed)\\n    np.random.seed(seed)\\n    torch.backends.cudnn.deterministic = True\\n    torch.backends.cudnn.benchmark = False\\n    lookback = int(options[\\"lookback\\"])\\n    values = np.log1p(train.to_numpy(dtype=np.float64))\\n    center = float(values.mean())\\n    scale = float(values.std())\\n    if scale == 0:\\n        scale = 1.0\\n    normalized = ((values - center) / scale).astype(np.float32)\\n    if len(normalized) <= lookback + 20:\\n        raise ValueError(\\"Không đủ train cho neural lookback.\\")\\n    x = np.stack([normalized[i-lookback:i] for i in range(lookback, len(normalized))])[:, :, None]\\n    y = normalized[lookback:, None]\\n    # Validation nội bộ là đoạn cuối train của fold, không dùng ngày sau origin.\\n    cut = max(1, int(len(x) * 0.8))\\n    if cut >= len(x):\\n        raise ValueError(\\"Không đủ cửa sổ validation nội bộ.\\")\\n    device = torch.device(\\"cuda\\")\\n\\n    class SequenceModel(nn.Module):\\n        def __init__(self):\\n            super().__init__()\\n            cell = nn.RNN if model_name == \\"rnn\\" else nn.LSTM\\n            self.recurrent = cell(1, int(options[\\"hidden_size\\"]), batch_first=True)\\n            self.head = nn.Linear(int(options[\\"hidden_size\\"]), 1)\\n\\n        def forward(self, batch):\\n            output, _ = self.recurrent(batch)\\n            return self.head(output[:, -1, :])\\n\\n    estimator = SequenceModel().to(device)\\n    optimizer = torch.optim.Adam(estimator.parameters(), lr=float(options[\\"learning_rate\\"]))\\n    criterion = nn.MSELoss()\\n    loader = DataLoader(\\n        TensorDataset(torch.from_numpy(x[:cut]), torch.from_numpy(y[:cut])),\\n        batch_size=int(options[\\"batch_size\\"]), shuffle=True,\\n        generator=torch.Generator().manual_seed(seed),\\n    )\\n    x_val = torch.from_numpy(x[cut:]).to(device)\\n    y_val = torch.from_numpy(y[cut:]).to(device)\\n    best = float(\\"inf\\")\\n    patience = 0\\n    best_state = None\\n    epochs_run = 0\\n    start = time.perf_counter()\\n    for epoch in range(int(options[\\"max_epochs\\"])):\\n        estimator.train()\\n        for bx, by in loader:\\n            bx, by = bx.to(device), by.to(device)\\n            optimizer.zero_grad(set_to_none=True)\\n            loss = criterion(estimator(bx), by)\\n            loss.backward()\\n            optimizer.step()\\n        estimator.eval()\\n        with torch.inference_mode():\\n            loss_val = float(criterion(estimator(x_val), y_val).item())\\n        epochs_run = epoch + 1\\n        if loss_val < best - float(options[\\"min_delta\\"]):\\n            best = loss_val\\n            best_state = {k: v.detach().cpu().clone() for k, v in estimator.state_dict().items()}\\n            patience = 0\\n        else:\\n            patience += 1\\n            if patience >= int(options[\\"patience\\"]):\\n                break\\n    estimator.load_state_dict(best_state)\\n    torch.cuda.synchronize()\\n    fit_seconds = time.perf_counter() - start\\n    # Đường fitted trên train chỉ dùng chẩn đoán, không là điểm validation.\\n    estimator.eval()\\n    with torch.inference_mode():\\n        fitted_scaled = []\\n        for offset in range(0, len(x), 512):\\n            fitted_scaled.extend(estimator(torch.from_numpy(x[offset:offset+512]).to(device)).cpu().numpy().ravel())\\n    fitted = np.expm1(np.asarray(fitted_scaled) * scale + center)\\n    history = list(normalized)\\n    prediction = []\\n    start = time.perf_counter()\\n    with torch.inference_mode():\\n        for _ in future:\\n            window = torch.tensor(history[-lookback:], dtype=torch.float32, device=device).view(1, lookback, 1)\\n            value_scaled = float(estimator(window).item())\\n            history.append(value_scaled)\\n            prediction.append(np.expm1(value_scaled * scale + center))\\n    torch.cuda.synchronize()\\n    artifact = {\\n        \\"state_dict\\": best_state, \\"model_name\\": model_name, \\"lookback\\": lookback,\\n        \\"hidden_size\\": int(options[\\"hidden_size\\"]), \\"center\\": center, \\"scale\\": scale,\\n    }\\n    return np.asarray(prediction), fitted, train.index[lookback:], artifact, {\\n        \\"input_set\\": \\"F0\\", \\"strategy\\": \\"recursive\\", \\"epochs\\": epochs_run,\\n        \\"internal_val_loss\\": best, \\"fit_seconds\\": fit_seconds,\\n        \\"predict_seconds\\": time.perf_counter() - start,\\n    }\\n\\n\\ndef fit_model(train, future, model_name, config):\\n    cloud_runtime()\\n    if model_name in TABULAR:\\n        return fit_tabular(train, future, model_name, config)\\n    if model_name in STATISTICAL:\\n        return fit_statistical(train, future, model_name, config)\\n    if model_name in NEURAL:\\n        return fit_neural(train, future, model_name, config)\\n    if model_name in {\\"snaive_year\\", \\"snaive_two_year\\"}:\\n        from cloud_pilot import baseline\\n        start = time.perf_counter()\\n        prediction = baseline(train, future, model_name)\\n        return prediction, None, None, {\\"method\\": model_name}, {\\n            \\"input_set\\": \\"F0\\", \\"strategy\\": \\"recursive\\", \\"fit_seconds\\": 0.0,\\n            \\"predict_seconds\\": time.perf_counter() - start,\\n        }\\n    raise ValueError(\\"Model chưa triển khai: \\" + model_name)\\n\\n\\n\\ndef make_screening_figure(train, actual, prediction, fitted, fitted_dates, path, title, fold):\\n    import matplotlib.pyplot as plt\\n    fig, axes = plt.subplots(2, 1, figsize=(12, 7))\\n    axes[0].plot(train.index, train.values, label=\\"Actual train\\", linewidth=0.8)\\n    if fitted is not None:\\n        axes[0].plot(fitted_dates, np.maximum(fitted, 0), label=\\"Fitted train\\", alpha=0.7, linewidth=0.7)\\n    axes[0].set_title(\\"Train - fitted chỉ dùng chẩn đoán\\")\\n    axes[1].plot(actual.index, actual.values, label=\\"Actual validation\\")\\n    axes[1].plot(actual.index, prediction, label=\\"Forecast từ origin cố định\\")\\n    axes[1].set_title(title + \\" - development \\" + fold)\\n    for axis in axes:\\n        axis.set_ylabel(\\"Giá trị tiền theo nguồn\\")\\n        axis.legend()\\n    fig.tight_layout()\\n    fig.savefig(path, dpi=180)\\n    plt.close(fig)\\n\\n\\ndef run(csv_path, manifest_path, config_path, output):\\n    runtime = cloud_runtime()\\n    data, source = load_snapshot(csv_path, manifest_path)\\n    config = json.loads(Path(config_path).read_text(encoding=\\"utf-8\\"))\\n    validate_config(config)\\n    if config[\\"runtime\\"] == \\"gpu\\":\\n        import torch\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\"Hãy chọn Colab GPU runtime trước khi chạy RNN/LSTM.\\")\\n    packages = {}\\n    for name in [\\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"statsmodels\\", \\"matplotlib\\", \\"joblib\\"]:\\n        packages[name] = importlib.metadata.version(name)\\n    optional_packages = {\\"random_forest\\": [], \\"extra_trees\\": [], \\"xgboost\\": [\\"xgboost\\"],\\n                         \\"lightgbm\\": [\\"lightgbm\\"], \\"catboost\\": [\\"catboost\\"],\\n                         \\"rnn\\": [\\"torch\\"], \\"lstm\\": [\\"torch\\"]}\\n    for model in config[\\"models\\"]:\\n        for name in optional_packages.get(model, []):\\n            packages[name] = importlib.metadata.version(name)\\n    identity = {\\n        \\"snapshot_sha256\\": source[\\"sha256\\"], \\"config_sha256\\": sha256(config_path),\\n        \\"code_sha256\\": sha256(__file__), \\"base_code_sha256\\": sha256(Path(__file__).with_name(\\"cloud_pilot.py\\")),\\n        \\"runtime\\": runtime, \\"python\\": platform.python_version(),\\n        \\"packages\\": packages, \\"machine\\": platform.machine(), \\"cpu_count\\": os.cpu_count(),\\n        \\"gpu_name\\": (torch.cuda.get_device_name(0) if config[\\"runtime\\"] == \\"gpu\\" else None),\\n    }\\n    run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]\\n    root = Path(output) / run_id\\n    root.mkdir(parents=True, exist_ok=True)\\n    write_json(root / \\"manifest.json\\", {\\"run_id\\": run_id, \\"source\\": source, \\"config\\": config, **identity})\\n    series_frame = data.set_index(\\"sale_date\\")\\n    results = []\\n    for fold in config[\\"folds\\"]:\\n        origin = pd.Timestamp(FOLDS[fold])\\n        for target in config[\\"targets\\"]:\\n            series = series_frame[\\"g\\" if target == \\"G\\" else \\"cogs\\"]\\n            train = series.loc[:origin].copy()\\n            for horizon in config[\\"horizons\\"]:\\n                future = pd.date_range(origin + pd.Timedelta(days=1), periods=horizon)\\n                actual = series.reindex(future)\\n                if actual.isna().any():\\n                    raise ValueError(\\"Thiếu actual validation.\\")\\n                for model in config[\\"models\\"]:\\n                    task = root / f\\"{fold}_{target}_{horizon}_{model}\\"\\n                    task.mkdir(exist_ok=True)\\n                    status_path = task / \\"status.json\\"\\n                    if status_path.exists():\\n                        old = json.loads(status_path.read_text(encoding=\\"utf-8\\"))\\n                        hashes = old.get(\\"artifact_sha256\\", {})\\n                        if old.get(\\"status\\") == \\"success\\" and hashes and all(\\n                            (task / name).exists() and sha256(task / name) == digest\\n                            for name, digest in hashes.items()\\n                        ):\\n                            results.append(old)\\n                            continue\\n                    record = {\\n                        \\"run_id\\": run_id, \\"snapshot_id\\": source[\\"snapshot_id\\"],\\n                        \\"model_id\\": model, \\"target\\": target, \\"fold_id\\": fold,\\n                        \\"origin\\": str(origin.date()), \\"horizon\\": horizon,\\n                        \\"split\\": \\"development\\", \\"seed\\": config[\\"seed\\"],\\n                    }\\n                    write_json(status_path, {**record, \\"status\\": \\"running\\"})\\n                    try:\\n                        raw, fitted, fitted_dates, estimator, timing = fit_model(train, future, model, config)\\n                        if raw.shape != (horizon,) or not np.isfinite(raw).all():\\n                            raise ValueError(\\"Forecast sai độ dài hoặc có NaN/Inf.\\")\\n                        prediction = np.maximum(raw, 0)\\n                        pd.DataFrame({\\n                            **{k: [v] * horizon for k, v in record.items()},\\n                            \\"ds\\": future, \\"lead_time\\": np.arange(1, horizon+1),\\n                            \\"y_true\\": actual.to_numpy(), \\"y_pred_raw\\": raw,\\n                            \\"y_pred\\": prediction,\\n                        }).to_csv(task / \\"predictions.csv\\", index=False)\\n                        if model in NEURAL:\\n                            import torch\\n                            torch.save(estimator, task / \\"model.pt\\")\\n                            model_file = \\"model.pt\\"\\n                        else:\\n                            import joblib\\n                            joblib.dump(estimator, task / \\"model.joblib\\")\\n                            model_file = \\"model.joblib\\"\\n                        artifacts = [\\"predictions.csv\\", model_file]\\n                        if fitted is not None:\\n                            pd.DataFrame({\\"ds\\": fitted_dates, \\"fitted_raw\\": fitted}).to_csv(\\n                                task / \\"fitted.csv\\", index=False)\\n                            artifacts.append(\\"fitted.csv\\")\\n                        make_screening_figure(train, actual, prediction, fitted, fitted_dates,\\n                                              task / \\"plot.png\\", f\\"{target} / {model} / H={horizon}\\", fold)\\n                        artifacts.append(\\"plot.png\\")\\n                        record.update(\\n                            status=\\"success\\", **score(actual.to_numpy(), prediction, train.to_numpy()),\\n                            **timing, clipped_negative=int((raw < 0).sum()),\\n                            artifact_sha256={name: sha256(task / name) for name in artifacts},\\n                        )\\n                    except Exception as exc:\\n                        record.update(status=\\"failed\\", error_type=type(exc).__name__, error=str(exc))\\n                    write_json(status_path, record)\\n                    results.append(record)\\n                    print(fold, target, horizon, model, record[\\"status\\"], flush=True)\\n                    pd.DataFrame(results).drop(columns=[\\"artifact_sha256\\"], errors=\\"ignore\\").to_csv(\\n                        root / \\"metrics.csv\\", index=False)\\n    write_json(root / \\"summary.json\\", {\\n        \\"total\\": len(results), \\"success\\": sum(r[\\"status\\"] == \\"success\\" for r in results),\\n        \\"failed\\": sum(r[\\"status\\"] != \\"success\\" for r in results),\\n    })\\n    return str(root)\\n", "configs/screening_cpu_pilot.json": "{\\n  \\"seed\\": 42,\\n  \\"targets\\": [\\n    \\"G\\",\\n    \\"COGS\\"\\n  ],\\n  \\"horizons\\": [\\n    30,\\n    548\\n  ],\\n  \\"lag_days\\": [\\n    1,\\n    2,\\n    3,\\n    7,\\n    14,\\n    28,\\n    30,\\n    60,\\n    90,\\n    180,\\n    365,\\n    730\\n  ],\\n  \\"rolling_days\\": [\\n    7,\\n    28,\\n    90,\\n    365\\n  ],\\n  \\"runtime\\": \\"cpu\\",\\n  \\"models\\": [\\n    \\"snaive_year\\",\\n    \\"snaive_two_year\\",\\n    \\"random_forest\\",\\n    \\"extra_trees\\",\\n    \\"xgboost\\",\\n    \\"lightgbm\\",\\n    \\"catboost\\",\\n    \\"arima\\",\\n    \\"arimax\\",\\n    \\"sarima\\",\\n    \\"sarimax\\"\\n  ],\\n  \\"random_forest\\": {\\n    \\"n_estimators\\": 120,\\n    \\"min_samples_leaf\\": 5,\\n    \\"n_jobs\\": 2\\n  },\\n  \\"extra_trees\\": {\\n    \\"n_estimators\\": 120,\\n    \\"min_samples_leaf\\": 5,\\n    \\"n_jobs\\": 2\\n  },\\n  \\"xgboost\\": {\\n    \\"n_estimators\\": 200,\\n    \\"max_depth\\": 4,\\n    \\"learning_rate\\": 0.05,\\n    \\"objective\\": \\"reg:squarederror\\",\\n    \\"tree_method\\": \\"hist\\",\\n    \\"n_jobs\\": 2\\n  },\\n  \\"lightgbm\\": {\\n    \\"n_estimators\\": 200,\\n    \\"num_leaves\\": 15,\\n    \\"learning_rate\\": 0.05,\\n    \\"min_child_samples\\": 30,\\n    \\"verbosity\\": -1,\\n    \\"n_jobs\\": 2,\\n    \\"deterministic\\": true,\\n    \\"force_col_wise\\": true\\n  },\\n  \\"catboost\\": {\\n    \\"iterations\\": 200,\\n    \\"depth\\": 5,\\n    \\"learning_rate\\": 0.05,\\n    \\"loss_function\\": \\"RMSE\\",\\n    \\"verbose\\": false,\\n    \\"thread_count\\": 2,\\n    \\"allow_writing_files\\": false\\n  },\\n  \\"statistical\\": {\\n    \\"arima\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"maxiter\\": 60\\n    },\\n    \\"arimax\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"maxiter\\": 60\\n    },\\n    \\"sarima\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"seasonal_order\\": [\\n        1,\\n        0,\\n        0,\\n        7\\n      ],\\n      \\"maxiter\\": 60\\n    },\\n    \\"sarimax\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"seasonal_order\\": [\\n        1,\\n        0,\\n        0,\\n        7\\n      ],\\n      \\"maxiter\\": 60\\n    }\\n  },\\n  \\"protocol_id\\": \\"screening-cpu-pilot-v1\\",\\n  \\"folds\\": [\\n    \\"F5\\"\\n  ]\\n}\\n", "configs/screening_cpu_cv.json": "{\\n  \\"seed\\": 42,\\n  \\"targets\\": [\\n    \\"G\\",\\n    \\"COGS\\"\\n  ],\\n  \\"horizons\\": [\\n    30,\\n    548\\n  ],\\n  \\"lag_days\\": [\\n    1,\\n    2,\\n    3,\\n    7,\\n    14,\\n    28,\\n    30,\\n    60,\\n    90,\\n    180,\\n    365,\\n    730\\n  ],\\n  \\"rolling_days\\": [\\n    7,\\n    28,\\n    90,\\n    365\\n  ],\\n  \\"runtime\\": \\"cpu\\",\\n  \\"models\\": [\\n    \\"snaive_year\\",\\n    \\"snaive_two_year\\",\\n    \\"random_forest\\",\\n    \\"extra_trees\\",\\n    \\"xgboost\\",\\n    \\"lightgbm\\",\\n    \\"catboost\\",\\n    \\"arima\\",\\n    \\"arimax\\",\\n    \\"sarima\\",\\n    \\"sarimax\\"\\n  ],\\n  \\"random_forest\\": {\\n    \\"n_estimators\\": 120,\\n    \\"min_samples_leaf\\": 5,\\n    \\"n_jobs\\": 2\\n  },\\n  \\"extra_trees\\": {\\n    \\"n_estimators\\": 120,\\n    \\"min_samples_leaf\\": 5,\\n    \\"n_jobs\\": 2\\n  },\\n  \\"xgboost\\": {\\n    \\"n_estimators\\": 200,\\n    \\"max_depth\\": 4,\\n    \\"learning_rate\\": 0.05,\\n    \\"objective\\": \\"reg:squarederror\\",\\n    \\"tree_method\\": \\"hist\\",\\n    \\"n_jobs\\": 2\\n  },\\n  \\"lightgbm\\": {\\n    \\"n_estimators\\": 200,\\n    \\"num_leaves\\": 15,\\n    \\"learning_rate\\": 0.05,\\n    \\"min_child_samples\\": 30,\\n    \\"verbosity\\": -1,\\n    \\"n_jobs\\": 2,\\n    \\"deterministic\\": true,\\n    \\"force_col_wise\\": true\\n  },\\n  \\"catboost\\": {\\n    \\"iterations\\": 200,\\n    \\"depth\\": 5,\\n    \\"learning_rate\\": 0.05,\\n    \\"loss_function\\": \\"RMSE\\",\\n    \\"verbose\\": false,\\n    \\"thread_count\\": 2,\\n    \\"allow_writing_files\\": false\\n  },\\n  \\"statistical\\": {\\n    \\"arima\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"maxiter\\": 60\\n    },\\n    \\"arimax\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"maxiter\\": 60\\n    },\\n    \\"sarima\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"seasonal_order\\": [\\n        1,\\n        0,\\n        0,\\n        7\\n      ],\\n      \\"maxiter\\": 60\\n    },\\n    \\"sarimax\\": {\\n      \\"order\\": [\\n        1,\\n        1,\\n        1\\n      ],\\n      \\"seasonal_order\\": [\\n        1,\\n        0,\\n        0,\\n        7\\n      ],\\n      \\"maxiter\\": 60\\n    }\\n  },\\n  \\"protocol_id\\": \\"screening-cpu-cv-v1\\",\\n  \\"folds\\": [\\n    \\"F1\\",\\n    \\"F2\\",\\n    \\"F3\\",\\n    \\"F4\\",\\n    \\"F5\\"\\n  ]\\n}\\n", "configs/screening_gpu_pilot.json": "{\\n  \\"seed\\": 42,\\n  \\"targets\\": [\\n    \\"G\\",\\n    \\"COGS\\"\\n  ],\\n  \\"horizons\\": [\\n    30,\\n    548\\n  ],\\n  \\"lag_days\\": [\\n    1,\\n    2,\\n    3,\\n    7,\\n    14,\\n    28,\\n    30,\\n    60,\\n    90,\\n    180,\\n    365,\\n    730\\n  ],\\n  \\"rolling_days\\": [\\n    7,\\n    28,\\n    90,\\n    365\\n  ],\\n  \\"runtime\\": \\"gpu\\",\\n  \\"models\\": [\\n    \\"snaive_year\\",\\n    \\"snaive_two_year\\",\\n    \\"rnn\\",\\n    \\"lstm\\"\\n  ],\\n  \\"neural\\": {\\n    \\"lookback\\": 90,\\n    \\"hidden_size\\": 32,\\n    \\"learning_rate\\": 0.001,\\n    \\"batch_size\\": 128,\\n    \\"max_epochs\\": 30,\\n    \\"patience\\": 5,\\n    \\"min_delta\\": 0.0001\\n  },\\n  \\"protocol_id\\": \\"screening-gpu-pilot-v1\\",\\n  \\"folds\\": [\\n    \\"F5\\"\\n  ]\\n}\\n", "configs/screening_gpu_cv.json": "{\\n  \\"seed\\": 42,\\n  \\"targets\\": [\\n    \\"G\\",\\n    \\"COGS\\"\\n  ],\\n  \\"horizons\\": [\\n    30,\\n    548\\n  ],\\n  \\"lag_days\\": [\\n    1,\\n    2,\\n    3,\\n    7,\\n    14,\\n    28,\\n    30,\\n    60,\\n    90,\\n    180,\\n    365,\\n    730\\n  ],\\n  \\"rolling_days\\": [\\n    7,\\n    28,\\n    90,\\n    365\\n  ],\\n  \\"runtime\\": \\"gpu\\",\\n  \\"models\\": [\\n    \\"snaive_year\\",\\n    \\"snaive_two_year\\",\\n    \\"rnn\\",\\n    \\"lstm\\"\\n  ],\\n  \\"neural\\": {\\n    \\"lookback\\": 90,\\n    \\"hidden_size\\": 32,\\n    \\"learning_rate\\": 0.001,\\n    \\"batch_size\\": 128,\\n    \\"max_epochs\\": 30,\\n    \\"patience\\": 5,\\n    \\"min_delta\\": 0.0001\\n  },\\n  \\"protocol_id\\": \\"screening-gpu-cv-v1\\",\\n  \\"folds\\": [\\n    \\"F1\\",\\n    \\"F2\\",\\n    \\"F3\\",\\n    \\"F4\\",\\n    \\"F5\\"\\n  ]\\n}\\n"}')
for relative, content in embedded_files.items():
    path = project / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
os.chdir(project)
sys.path.insert(0, str(project))
print("Đã chuẩn bị", len(embedded_files), "file code và cấu hình trên VM Colab.")


## 3. Chọn snapshot actual đã export trong Drive

Nếu có nhiều release, điền SNAPSHOT_FOLDER. Không đưa token vào notebook này.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
storage = Path("/content/drive/MyDrive/retail-sales-forecasting-research")
storage.mkdir(parents=True, exist_ok=True)
candidates = sorted(storage.glob("snapshots/*/manifest.json"))
for item in candidates:
    print("Snapshot:", item.parent.name)
if not candidates:
    raise FileNotFoundError("Chưa có snapshot trong Drive; chạy notebook 01 để export actual trước.")
SNAPSHOT_FOLDER = ""  # Nếu có nhiều snapshot, điền tên thư mục release đã chọn.
if SNAPSHOT_FOLDER:
    manifest_path = storage / "snapshots" / SNAPSHOT_FOLDER / "manifest.json"
elif len(candidates) == 1:
    manifest_path = candidates[0]
else:
    raise ValueError("Có nhiều snapshot; điền SNAPSHOT_FOLDER để chọn đúng release.")
csv_path = manifest_path.with_name("actual.csv")
from cloud_pilot import load_snapshot
data, snapshot_manifest = load_snapshot(csv_path, manifest_path)
print("Đã đối chiếu snapshot:", snapshot_manifest["snapshot_id"],
      "| release:", snapshot_manifest["release_id"], "| số ngày:", len(data))


## 4. Chạy model

MODE=pilot chạy F5 trước. MODE=cv chạy đủ năm rolling-origin fold; có thể mất nhiều phiên Colab. Mỗi task được lưu vào Drive và resume khi chạy lại cùng snapshot/config/code.

In [ ]:
from pathlib import Path
from screening import run
MODE = "pilot"  # "pilot": F5; "cv": F1-F5. Chỉ đổi sang cv sau khi pilot ổn.
config_path = project / "configs" / ("screening_gpu_" + MODE + ".json")
result_dir = run(csv_path, manifest_path, config_path, storage / "screening_runs")
print("Kết quả:", result_dir)


## 5. Xem đủ trạng thái và điểm

MAE thấp hơn là tốt hơn trong cùng target/horizon. Model failed không được xóa khỏi báo cáo. Điểm development chưa là điểm final test.

In [ ]:
import json
import pandas as pd
from IPython.display import display
result_root = Path(result_dir)
print(json.loads((result_root / "summary.json").read_text(encoding="utf-8")))
metrics = pd.read_csv(result_root / "metrics.csv")
print("Trạng thái từng model:")
display(metrics[["fold_id", "target", "horizon", "model_id", "status"]])
failed = metrics[metrics["status"] != "success"]
if len(failed):
    print("Model lỗi; không được bỏ khỏi báo cáo:")
    columns = [c for c in ["fold_id", "target", "horizon", "model_id", "error_type", "error"] if c in failed]
    display(failed[columns])
else:
    print("Không có model lỗi.")
success = metrics[metrics["status"] == "success"].copy()
expected_folds = len(json.loads(config_path.read_text(encoding="utf-8"))["folds"])
if len(success):
    ranked = (success.groupby(["target", "horizon", "model_id"], as_index=False)
              .agg(folds=("fold_id", "nunique"), mae=("mae", "mean"),
                   mase=("mase", "mean"), mape=("mape", "mean"), mape_coverage=("mape_coverage", "mean"),
                   rse=("rse", "mean"), fit_seconds=("fit_seconds", "mean"),
                   predict_seconds=("predict_seconds", "mean")))
    ranked["complete"] = ranked["folds"] == expected_folds
    print("MAE trung bình theo fold; chỉ xếp hạng dòng complete=True:")
    display(ranked.sort_values(["target", "horizon", "complete", "mae"],
                               ascending=[True, True, False, True]))
else:
    print("Chưa có model thành công để xếp hạng.")


## 6. Vẽ actual và top 5 model F5

Biểu đồ chỉ vẽ model success tại F5; bảng phía trên vẫn hiện toàn bộ fold và lỗi.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
for target in ("G", "COGS"):
    for horizon in (30, 548):
        subset = success[(success["target"] == target) &
                         (success["horizon"] == horizon) &
                         (success["fold_id"] == "F5")].sort_values("mae").head(5)
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(13, 4))
        for row in subset.itertuples():
            path = result_root / f"F5_{target}_{horizon}_{row.model_id}" / "predictions.csv"
            predictions = pd.read_csv(path, parse_dates=["ds"])
            ax.plot(predictions["ds"], predictions["y_pred"],
                    label=f"{row.model_id} | MAE {row.mae:,.0f}", alpha=0.8)
        first = pd.read_csv(result_root / f"F5_{target}_{horizon}_{subset.iloc[0].model_id}" /
                            "predictions.csv", parse_dates=["ds"])
        ax.plot(first["ds"], first["y_true"], color="black", label="Actual", linewidth=2)
        ax.set_title(f"{target} | H={horizon} | F5 development - Top 5 theo MAE")
        ax.set_ylabel("Giá trị tiền theo nguồn")
        ax.legend()
        ax.grid(alpha=0.2)
        fig.tight_layout()
        display(fig)
        plt.close(fig)
